# CardioPredict — Notebook d'exploration et de visualisation

Ce notebook présente l'analyse exploratoire des deux jeux de données utilisés dans le projet **CardioPredict**.

## Objectifs
1. Comprendre la structure et les dimensions des deux datasets ;
2. Décrire les variables et détecter les anomalies éventuelles ;
3. Visualiser les distributions et les relations entre variables ;
4. Dégager des premières interprétations avant la phase de modélisation.

## Datasets
| Dataset | Source | Description |
|---------|--------|-------------|
| `cardio.csv` | Kaggle — Sulianova | Dataset principal — facteurs généraux et mode de vie |
| `heart.csv` | Kaggle — johnsmith88 | Dataset secondaire — variables cliniques spécialisées |

> ⚠️ Placer `cardio.csv` et `heart.csv` dans le même dossier que ce notebook avant exécution.

## 1. Import des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

plt.rcParams["figure.dpi"]     = 110
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False

# Palette cohérente avec le site CardioPredict
BLUE   = "#1F5FBF"
RED    = "#EF4444"
GREEN  = "#10B981"
MUTED  = "#5E7591"
SOFT   = "#EAF2FF"

print("Imports OK")

## 2. Chargement et nettoyage initial des données

Les fichiers CSV utilisent le séparateur **point-virgule** `;`.  
Deux corrections sont appliquées immédiatement :
- `weight` dans `cardio` peut contenir des **virgules décimales** → conversion en point ;
- `oldpeak` dans `heart` utilise une **virgule décimale** → idem.

Le dataset Heart contient en outre **723 lignes dupliquées sur 1 025** — elles sont supprimées  
pour éviter tout *data leakage* lors de la modélisation.

In [ ]:
cardio = pd.read_csv("cardio.csv", sep=";")
heart  = pd.read_csv("heart.csv",  sep=";")

# Correction des décimales
cardio["weight"] = (cardio["weight"].astype(str)
                    .str.replace(",", ".", regex=False)
                    .pipe(pd.to_numeric, errors="coerce"))

heart["oldpeak"] = (heart["oldpeak"].astype(str)
                    .str.replace(",", ".", regex=False)
                    .pipe(pd.to_numeric, errors="coerce"))

# Suppression des colonnes parasites
for col in ["dataset_id", "id"]:
    if col in cardio.columns: cardio = cardio.drop(columns=col)
for col in ["cardio_id", "dataset_id", "id_heart", "id"]:
    if col in heart.columns:  heart  = heart.drop(columns=col)

# Déduplication Heart
n_avant = len(heart)
heart = heart.drop_duplicates().reset_index(drop=True)
n_apres = len(heart)
print(f"Heart : {n_avant} lignes brutes → {n_apres} après déduplication ({n_avant - n_apres} doublons supprimés)")
print(f"Cardio : {cardio.shape}")
print(f"Heart  : {heart.shape}")

In [ ]:
print("── Aperçu cardio ──")
display(cardio.head(3))
print("\n── Aperçu heart ──")
display(heart.head(3))

## 3. Dimensions des datasets

In [ ]:
summary = pd.DataFrame({
    "Lignes":    [cardio.shape[0], heart.shape[0]],
    "Colonnes":  [cardio.shape[1], heart.shape[1]],
    "Cible":     ["cardio (0=malade, 1=malade)", "target (1=sain, 0=malade)"],
    "Type":      ["Facteurs généraux + mode de vie", "Variables cliniques spécialisées"]
}, index=["cardio","heart"])
display(summary)

## 4. Types de variables et valeurs manquantes

On vérifie le type de chaque colonne et la présence de valeurs manquantes.  
C'est une étape indispensable avant toute modélisation.

In [ ]:
print("=== cardio.info() ===")
cardio.info()
print("\n=== heart.info() ===")
heart.info()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, df, name in [(axes[0], cardio, "cardio"), (axes[1], heart, "heart")]:
    miss = df.isna().sum()
    if miss.sum() == 0:
        ax.text(0.5, 0.5, "Aucune valeur manquante ✓",
                ha="center", va="center", fontsize=13, color=GREEN,
                transform=ax.transAxes)
        ax.set_title(f"Valeurs manquantes — {name}")
        ax.axis("off")
    else:
        miss[miss > 0].plot(kind="barh", ax=ax, color=RED)
        ax.set_title(f"Valeurs manquantes — {name}")

plt.tight_layout()
plt.show()

## 5. Statistiques descriptives

Les statistiques permettent de repérer des valeurs aberrantes :  
valeurs minimales/maximales incohérentes, écarts-types anormalement grands, etc.

In [ ]:
print("── Statistiques cardio ──")
display(cardio.describe(include="all").transpose().round(2))

In [ ]:
print("── Statistiques heart ──")
display(heart.describe(include="all").transpose().round(2))

## 6. Préparation de variables dérivées

Dans `cardio`, l'âge est stocké en **jours** — peu lisible pour l'analyse.  
On crée une version en années et l'**IMC** (Indice de Masse Corporelle).

In [ ]:
cardio["age_years"] = cardio["age"] / 365.25
cardio["bmi"] = cardio["weight"] / ((cardio["height"] / 100) ** 2)
cardio["pulse_pressure"] = cardio["ap_hi"] - cardio["ap_lo"]
cardio["mean_arterial_pressure"] = cardio["ap_lo"] + (cardio["ap_hi"] - cardio["ap_lo"]) / 3

print("Plage d'?ge cardio :", round(cardio["age_years"].min(), 1), "?", round(cardio["age_years"].max(), 1), "ans")
print("IMC moyen :", round(cardio["bmi"].mean(), 2))
print("Pression puls?e moyenne :", round(cardio["pulse_pressure"].mean(), 2), "mmHg")
print("Pression art?rielle moyenne :", round(cardio["mean_arterial_pressure"].mean(), 2), "mmHg")
display(cardio[["age", "age_years", "height", "weight", "bmi", "ap_hi", "ap_lo", "pulse_pressure", "mean_arterial_pressure"]].head(5))


## 7. Répartition de la variable cible

Un déséquilibre important entre les classes (ex. 90% / 10%) biaiserait les modèles —  
il faudrait alors appliquer un rééquilibrage (SMOTE, sous-échantillonnage...).  

> ℹ️ Rappel sur l'encodage Heart : `target = 1` = **sain**, `target = 0` = **malade**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Cardio
counts_c = cardio["cardio"].value_counts().sort_index()
axes[0].bar(["Sain (0)", "Malade (1)"], counts_c.values, color=[GREEN, RED], alpha=0.85)
for i, v in enumerate(counts_c.values):
    axes[0].text(i, v + 400, f"{v:,}\n({v/len(cardio)*100:.1f}%)", ha="center", fontsize=10)
axes[0].set_title("Répartition cible — Cardio")
axes[0].set_ylabel("Nombre d'observations")

# Heart
counts_h = heart["target"].value_counts().sort_index()
axes[1].bar(["Malade (0)", "Sain (1)"], counts_h.values, color=[RED, GREEN], alpha=0.85)
for i, v in enumerate(counts_h.values):
    axes[1].text(i, v + 3, f"{v}\n({v/len(heart)*100:.1f}%)", ha="center", fontsize=10)
axes[1].set_title("Répartition cible — Heart")
axes[1].set_ylabel("Nombre d'observations")

plt.tight_layout()
plt.show()

print(f"Cardio — équilibre : {counts_c[0]/len(cardio)*100:.1f}% / {counts_c[1]/len(cardio)*100:.1f}%")
print(f"Heart  — équilibre : {counts_h[0]/len(heart)*100:.1f}% / {counts_h[1]/len(heart)*100:.1f}%")

### Interprétation
Les deux datasets présentent un équilibre satisfaisant entre les classes (~50/50),  
ce qui est favorable pour la modélisation : **aucun rééquilibrage artificiel n'est nécessaire**.

## 8. Distribution de l'âge

L'âge est l'un des principaux facteurs de risque cardiovasculaire connu médicalement.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(cardio.loc[cardio["cardio"]==0, "age_years"], bins=30,
             alpha=0.7, color=GREEN, label="Sain")
axes[0].hist(cardio.loc[cardio["cardio"]==1, "age_years"], bins=30,
             alpha=0.7, color=RED, label="Malade")
axes[0].set_title("Distribution de l'âge — Cardio")
axes[0].set_xlabel("Âge (années)")
axes[0].set_ylabel("Effectif")
axes[0].legend()

axes[1].hist(heart.loc[heart["target"]==1, "age"], bins=25,
             alpha=0.7, color=GREEN, label="Sain (target=1)")
axes[1].hist(heart.loc[heart["target"]==0, "age"], bins=25,
             alpha=0.7, color=RED, label="Malade (target=0)")
axes[1].set_title("Distribution de l'âge — Heart")
axes[1].set_xlabel("Âge (années)")
axes[1].set_ylabel("Effectif")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Âge moyen sains   (cardio) : {cardio.loc[cardio['cardio']==0,'age_years'].mean():.1f} ans")
print(f"Âge moyen malades (cardio) : {cardio.loc[cardio['cardio']==1,'age_years'].mean():.1f} ans")

### Interprétation
On observe que les patients malades sont en moyenne **plus âgés** dans les deux datasets.  
C'est cohérent avec la littérature médicale : le risque cardiovasculaire augmente avec l'âge.

## 9. Répartition selon le sexe

> Note encodage : dans `cardio`, `gender = 1` = femme, `gender = 2` = homme.  
> Dans `heart`, `sex = 0` = femme, `sex = 1` = homme.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Cardio
cardio["gender_label"] = cardio["gender"].map({1: "Femme", 2: "Homme"})
vc_c = cardio.groupby("gender_label")["cardio"].value_counts(normalize=True).unstack()
vc_c.plot(kind="bar", ax=axes[0], color=[GREEN, RED], alpha=0.85)
axes[0].set_title("Taux de maladie par sexe — Cardio")
axes[0].set_xlabel("Genre")
axes[0].set_ylabel("Proportion")
axes[0].tick_params(axis="x", rotation=0)
axes[0].legend(["Sain (0)", "Malade (1)"])

# Heart
heart["sex_label"] = heart["sex"].map({0: "Femme", 1: "Homme"})
vc_h = heart.groupby("sex_label")["target"].value_counts(normalize=True).unstack()
vc_h.plot(kind="bar", ax=axes[1], color=[RED, GREEN], alpha=0.85)
axes[1].set_title("Taux par sexe — Heart (1=sain)")
axes[1].set_xlabel("Sexe")
axes[1].set_ylabel("Proportion")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(["Malade (0)", "Sain (1)"])

plt.tight_layout()
plt.show()

## 10. Cholestérol et maladie cardiovasculaire

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Cardio — cholesterol encodé en niveaux (1/2/3)
chol_labels = {1: "Normal", 2: "Élevé", 3: "Très élevé"}
cardio["chol_label"] = cardio["cholesterol"].map(chol_labels)
ct = pd.crosstab(cardio["chol_label"], cardio["cardio"], normalize="index")
ct.columns = ["Sain", "Malade"]
ct.reindex(["Normal","Élevé","Très élevé"]).plot(kind="bar", ax=axes[0],
                                                  color=[GREEN, RED], alpha=0.85)
axes[0].set_title("Taux de maladie par niveau de cholestérol — Cardio")
axes[0].set_xlabel("Niveau cholestérol")
axes[0].set_ylabel("Proportion")
axes[0].tick_params(axis="x", rotation=15)
axes[0].legend()

# Heart — cholestérol sérique continu
axes[1].boxplot([
    heart.loc[heart["target"]==0, "chol"].dropna(),
    heart.loc[heart["target"]==1, "chol"].dropna()
], labels=["Malade (0)", "Sain (1)"],
   patch_artist=True,
   boxprops=dict(facecolor=SOFT))
axes[1].set_title("Cholestérol sérique vs cible — Heart")
axes[1].set_ylabel("chol (mg/dL)")

plt.tight_layout()
plt.show()

### Interprétation
Dans `cardio`, les individus avec un cholestérol **très élevé** (niveau 3) présentent un taux de maladie  
sensiblement plus important. Dans `heart`, la distribution du cholestérol est comparable entre malades et sains — 
ce qui illustre que cette variable, seule, n'est pas suffisante pour prédire la maladie.

## 11. Pression artérielle et maladie

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Boxplot systolique
axes[0].boxplot([
    cardio.loc[cardio["cardio"]==0, "ap_hi"].clip(0, 250),
    cardio.loc[cardio["cardio"]==1, "ap_hi"].clip(0, 250)
], labels=["Sain (0)", "Malade (1)"], patch_artist=True,
   boxprops=dict(facecolor=SOFT))
axes[0].set_title("Pression systolique — Cardio")
axes[0].set_ylabel("ap_hi (mmHg)")

# Boxplot diastolique
axes[1].boxplot([
    cardio.loc[cardio["cardio"]==0, "ap_lo"].clip(0, 180),
    cardio.loc[cardio["cardio"]==1, "ap_lo"].clip(0, 180)
], labels=["Sain (0)", "Malade (1)"], patch_artist=True,
   boxprops=dict(facecolor=SOFT))
axes[1].set_title("Pression diastolique — Cardio")
axes[1].set_ylabel("ap_lo (mmHg)")

# Scatter ap_hi vs ap_lo
sample = cardio.sample(3000, random_state=42)
colors = sample["cardio"].map({0: GREEN, 1: RED})
axes[2].scatter(sample["ap_hi"].clip(0,250), sample["ap_lo"].clip(0,180),
                c=colors, alpha=0.35, s=8)
axes[2].set_title("ap_hi vs ap_lo (échantillon 3 000)")
axes[2].set_xlabel("ap_hi (systolique)")
axes[2].set_ylabel("ap_lo (diastolique)")
p1 = mpatches.Patch(color=GREEN, label="Sain")
p2 = mpatches.Patch(color=RED,   label="Malade")
axes[2].legend(handles=[p1, p2])

plt.tight_layout()
plt.show()

### Interprétation
La **pression systolique** (ap_hi) est clairement plus élevée chez les malades — c'est le facteur  
de risque cardiovasculaire le mieux documenté médicalement. La pression diastolique suit la même tendance.  
Le nuage de points confirme une séparation partielle mais réelle entre les deux classes.

## 12. Mode de vie : tabac, alcool, activité physique

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, title in [
    (axes[0], "smoke",  "Tabagisme vs cardio"),
    (axes[1], "alco",   "Alcool vs cardio"),
    (axes[2], "active", "Activité physique vs cardio")
]:
    ct = pd.crosstab(cardio[col], cardio["cardio"], normalize="index")
    ct.columns = ["Sain", "Malade"]
    ct.index = ["Non", "Oui"]
    ct.plot(kind="bar", ax=ax, color=[GREEN, RED], alpha=0.85)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Proportion")
    ax.tick_params(axis="x", rotation=0)
    ax.legend(loc="upper right")
    ax.set_ylim(0, 0.8)

plt.tight_layout()
plt.show()

### Interprétation
- **Tabac et alcool** : l'effet sur la maladie est présent mais modéré dans ce dataset,  
  probablement parce que la prévalence des fumeurs déclarés est faible (déclaratif).
- **Activité physique** : les personnes actives présentent un taux de maladie légèrement plus faible —  
  cohérent avec les recommandations médicales.

## 13. IMC et maladie cardiovasculaire

In [ ]:
# Filtrage des IMC aberrants (valeurs extrêmes)
cardio_clean = cardio[(cardio["bmi"] > 10) & (cardio["bmi"] < 60)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].boxplot([
    cardio_clean.loc[cardio_clean["cardio"]==0, "bmi"],
    cardio_clean.loc[cardio_clean["cardio"]==1, "bmi"]
], labels=["Sain (0)", "Malade (1)"], patch_artist=True,
   boxprops=dict(facecolor=SOFT))
axes[0].set_title("IMC selon la maladie — Cardio")
axes[0].set_ylabel("IMC (kg/m²)")

# Distribution par classe
axes[1].hist(cardio_clean.loc[cardio_clean["cardio"]==0, "bmi"],
             bins=40, alpha=0.7, color=GREEN, label="Sain", density=True)
axes[1].hist(cardio_clean.loc[cardio_clean["cardio"]==1, "bmi"],
             bins=40, alpha=0.7, color=RED, label="Malade", density=True)
axes[1].set_title("Distribution de l'IMC par classe")
axes[1].set_xlabel("IMC (kg/m²)")
axes[1].set_ylabel("Densité")
axes[1].legend()
axes[1].axvline(25, color="black", linestyle="--", alpha=0.5, label="Surpoids (25)")
axes[1].axvline(30, color="gray",  linestyle="--", alpha=0.5, label="Obésité (30)")

plt.tight_layout()
plt.show()

print(f"IMC médian sains   : {cardio_clean.loc[cardio_clean['cardio']==0,'bmi'].median():.1f}")
print(f"IMC médian malades : {cardio_clean.loc[cardio_clean['cardio']==1,'bmi'].median():.1f}")

### Interprétation
Les patients malades présentent un **IMC médian légèrement plus élevé**,  
ce qui confirme que le surpoids est un facteur de risque cardiovasculaire.  
La distribution est toutefois chevauchante : l'IMC seul n'est pas suffisant pour prédire la maladie.

## 14. Analyse du dataset Heart

Le dataset Heart contient des variables **cliniques plus spécialisées**.  
Rappel encodage cible : `target = 1` = sain, `target = 0` = malade.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Type de douleur thoracique
cp_labels = {0: "Angine typique", 1: "Atypique", 2: "Non angineux", 3: "Asympt."}
heart["cp_label"] = heart["cp"].map(cp_labels)
ct_cp = pd.crosstab(heart["cp_label"], heart["target"], normalize="index")
ct_cp.columns = ["Malade (0)", "Sain (1)"]
ct_cp.plot(kind="bar", ax=axes[0,0], color=[RED, GREEN], alpha=0.85)
axes[0,0].set_title("Type de douleur thoracique vs target")
axes[0,0].set_xlabel("cp")
axes[0,0].tick_params(axis="x", rotation=20)
axes[0,0].legend()

# Angine à l'effort
axes[0,1].bar(["Sans angine (0)", "Avec angine (1)"],
              [heart.loc[heart["exang"]==0,"target"].eq(0).mean(),
               heart.loc[heart["exang"]==1,"target"].eq(0).mean()],
              color=[BLUE, RED], alpha=0.85)
axes[0,1].set_title("Taux de maladie selon l'angine à l'effort")
axes[0,1].set_ylabel("Proportion malades (target=0)")
axes[0,1].set_ylim(0, 1)

# Fréquence cardiaque max
axes[1,0].boxplot([
    heart.loc[heart["target"]==0, "thalach"],
    heart.loc[heart["target"]==1, "thalach"]
], labels=["Malade (0)", "Sain (1)"], patch_artist=True,
   boxprops=dict(facecolor=SOFT))
axes[1,0].set_title("Fréquence cardiaque max vs target")
axes[1,0].set_ylabel("thalach (bpm)")

# Oldpeak (dépression ST)
axes[1,1].boxplot([
    heart.loc[heart["target"]==0, "oldpeak"],
    heart.loc[heart["target"]==1, "oldpeak"]
], labels=["Malade (0)", "Sain (1)"], patch_artist=True,
   boxprops=dict(facecolor=SOFT))
axes[1,1].set_title("Dépression ST (oldpeak) vs target")
axes[1,1].set_ylabel("oldpeak")

plt.suptitle("Variables cliniques du dataset Heart", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### Interprétation
- **Type de douleur thoracique** (`cp`) : les patients asymptomatiques ont paradoxalement plus de maladie détectée — car ils viennent en consultation pour autre chose et la maladie est découverte à l'examen.
- **Angine à l'effort** (`exang`) : les patients **sans** angine à l'effort présentent un taux de maladie plus élevé — une donnée contre-intuitive liée à l'encodage particulier de ce dataset.
- **Fréquence cardiaque max** (`thalach`) : les patients **sains** ont une fréquence maximale plus élevée, signe d'un meilleur état cardiovasculaire.
- **Oldpeak** : une dépression ST plus marquée est associée à la présence de maladie.

## 15. Matrices de corrélation

Les matrices de corrélation permettent d'identifier :
- les variables les plus corrélées à la cible ;
- les redondances entre variables (multicolinéarité).

On utilise le coefficient de Pearson sur les variables numériques.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Corrélation cardio
corr_c = cardio.drop(columns=["gender_label","chol_label"], errors="ignore").corr(numeric_only=True)
im0 = axes[0].imshow(corr_c, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
axes[0].set_xticks(range(len(corr_c.columns)))
axes[0].set_xticklabels(corr_c.columns, rotation=90, fontsize=9)
axes[0].set_yticks(range(len(corr_c.columns)))
axes[0].set_yticklabels(corr_c.columns, fontsize=9)
axes[0].set_title("Matrice de corrélation — Cardio")
plt.colorbar(im0, ax=axes[0], shrink=0.8)

# Corrélation heart
corr_h = heart.drop(columns=["sex_label","cp_label"], errors="ignore").corr(numeric_only=True)
im1 = axes[1].imshow(corr_h, cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
axes[1].set_xticks(range(len(corr_h.columns)))
axes[1].set_xticklabels(corr_h.columns, rotation=90, fontsize=9)
axes[1].set_yticks(range(len(corr_h.columns)))
axes[1].set_yticklabels(corr_h.columns, fontsize=9)
axes[1].set_title("Matrice de corrélation — Heart")
plt.colorbar(im1, ax=axes[1], shrink=0.8)

plt.tight_layout()
plt.show()

In [ ]:
# Top corrélations avec la cible
print("── Top corrélations avec 'cardio' ──")
corr_target_c = corr_c["cardio"].drop("cardio").abs().sort_values(ascending=False)
display(corr_target_c.head(8).rename("|corrélation|").round(4).to_frame())

print("\n── Top corrélations avec 'target' (Heart) ──")
corr_target_h = corr_h["target"].drop("target").abs().sort_values(ascending=False)
display(corr_target_h.head(8).rename("|corrélation|").round(4).to_frame())

### Interprétation
Les corrélations linéaires les plus fortes avec la cible `cardio` sont :
la **pression systolique** (ap_hi), le **cholestérol**, le **glucose** et l'**âge**.  

Pour `heart`, c'est la **fréquence cardiaque max** (thalach), le **type de douleur** (cp)  
et la **dépression ST** (oldpeak) qui sont les plus corrélés.  

> Ces corrélations sont à nuancer : les relations non-linéaires (captées par les forêts aléatoires)  
> peuvent être plus importantes que ce que Pearson révèle.

## 16. Détection des valeurs aberrantes

Le dataset Cardio contient des valeurs physiologiquement impossibles  
dans les colonnes de pression artérielle.

In [ ]:
quality_checks = pd.DataFrame({
    "Contr?le": [
        "ap_hi < ap_lo",
        "ap_hi < 60 mmHg",
        "ap_hi > 250 mmHg",
        "ap_lo < 40 mmHg",
        "ap_lo > 180 mmHg",
        "height < 100 cm",
        "height > 220 cm",
        "weight < 30 kg",
        "weight > 200 kg",
        "IMC < 12",
        "IMC > 60",
        "Pression puls?e < 10",
        "Pression puls?e > 120",
    ],
    "Nombre de lignes": [
        int((cardio["ap_hi"] < cardio["ap_lo"]).sum()),
        int((cardio["ap_hi"] < 60).sum()),
        int((cardio["ap_hi"] > 250).sum()),
        int((cardio["ap_lo"] < 40).sum()),
        int((cardio["ap_lo"] > 180).sum()),
        int((cardio["height"] < 100).sum()),
        int((cardio["height"] > 220).sum()),
        int((cardio["weight"] < 30).sum()),
        int((cardio["weight"] > 200).sum()),
        int((cardio["bmi"] < 12).sum()),
        int((cardio["bmi"] > 60).sum()),
        int((cardio["pulse_pressure"] < 10).sum()),
        int((cardio["pulse_pressure"] > 120).sum()),
    ]
})
quality_checks["% du dataset"] = (quality_checks["Nombre de lignes"] / len(cardio) * 100).round(3)
display(quality_checks)

suspect_mask = (
    (cardio["ap_hi"] < cardio["ap_lo"]) |
    (cardio["ap_hi"] < 60) | (cardio["ap_hi"] > 250) |
    (cardio["ap_lo"] < 40) | (cardio["ap_lo"] > 180) |
    (cardio["height"] < 100) | (cardio["height"] > 220) |
    (cardio["weight"] < 30) | (cardio["weight"] > 200) |
    (cardio["bmi"] < 12) | (cardio["bmi"] > 60)
)

print(f"Lignes suspectes au total : {suspect_mask.sum()} / {len(cardio)} ({suspect_mask.mean()*100:.2f} %)")
display(cardio.loc[suspect_mask, ["age_years", "height", "weight", "bmi", "ap_hi", "ap_lo", "pulse_pressure", "cardio"]].head(10))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(data=cardio, y="ap_hi", ax=axes[0], color=BLUE)
axes[0].set_title("Outliers pression systolique")
axes[0].set_ylabel("ap_hi")

sns.boxplot(data=cardio, y="ap_lo", ax=axes[1], color=RED)
axes[1].set_title("Outliers pression diastolique")
axes[1].set_ylabel("ap_lo")

sns.boxplot(data=cardio, y="bmi", ax=axes[2], color=GREEN)
axes[2].set_title("Outliers IMC")
axes[2].set_ylabel("IMC")

plt.tight_layout()
plt.show()


### Lecture du test des valeurs aberrantes

Ce test sert a verifier si certaines mesures sont medicalement ou statistiquement incoherentes avant la modelisation.  
Le tableau compte les lignes suspectes : tension systolique inferieure a la diastolique, pressions extremes, tailles ou poids improbables, IMC tres faible ou tres eleve.

Le resultat montre que ces anomalies existent, surtout sur la tension arterielle. Elles restent minoritaires par rapport aux 70 000 observations, mais elles doivent etre signalees car elles peuvent influencer les modeles. On ne les supprime pas automatiquement ici : on les documente, puis on controle leur impact dans le notebook de modelisation avec les erreurs, les faux negatifs et les faux positifs.


### Interpretation

Les valeurs aberrantes les plus sensibles concernent la pression arterielle : certaines lignes presentent des valeurs extremes ou incoherentes (`ap_hi < ap_lo`).  
Ces observations sont peu nombreuses par rapport aux 70 000 lignes, mais elles peuvent influencer les arbres de decision et les modeles sensibles aux distributions.

Decision retenue pour la modelisation :
- conserver une version **brute controlee** pour rester proche du dataset original ;
- documenter explicitement les incoherences ;
- utiliser une imputation mediane et une standardisation dans le pipeline ;
- comparer les faux negatifs/faux positifs pour verifier que ces valeurs n'entrainent pas de biais majeur.

Une amelioration possible serait de tester un scenario filtre, mais il n'est pas retenu comme flux principal afin de ne pas modifier artificiellement le dataset de reference.


## 17. Comparaison synthétique des deux datasets

In [ ]:
comparison = pd.DataFrame({
    "Lignes brutes":         [70000, 1025],
    "Lignes après nettoyage":[len(cardio), len(heart)],
    "Colonnes features":     [len(cardio.columns)-3, len(heart.columns)-1],
    "Variable cible":        ["cardio (0=sain, 1=malade)", "target (1=sain, 0=malade)"],
    "Type d'infos":          ["Facteurs généraux + mode de vie", "Variables cliniques précises"],
    "Accessible sans bilan": ["Oui", "Non — bilan cardiologique requis"],
    "Modèle retenu":         ["Random Forest (AUC 0.798)", "Rég. Logistique (AUC 0.871)"],
}, index=["cardio", "heart"])
display(comparison.T)

## 18. Conclusion

Cette analyse exploratoire met en evidence plusieurs resultats cles :

### Facteurs de risque identifies
| Variable | Dataset | Effet observe |
|----------|---------|---------------|
| Age | Cardio + Heart | Plus eleve chez les malades |
| Pression systolique (ap_hi) | Cardio | Forte correlation avec la maladie |
| IMC | Cardio | Legerement plus eleve chez les malades |
| Cholesterol eleve | Cardio | Associe a un taux de maladie plus important |
| Frequence cardiaque max | Heart | Plus basse chez les malades |
| Depression ST (oldpeak) | Heart | Plus marquee chez les malades |

### Qualite des donnees
- Le dataset **Heart** contient de nombreux doublons : ils doivent etre supprimes avant le split train/test pour eviter le data leakage.
- Le dataset **Cardio** contient des valeurs aberrantes sur la tension, la taille, le poids et l'IMC.
- Les incoherences les plus critiques sont les cas ou `ap_hi < ap_lo`, car elles contredisent la logique medicale de base.
- Ces anomalies restent minoritaires, mais elles doivent etre mentionnees dans l'evaluation ML.

### Choix des datasets
- **`cardio`** est retenu comme dataset **principal** : 70 000 observations, variables accessibles sans bilan medical.
- **`heart`** est le dataset **secondaire** : moins d'observations mais variables cliniques plus precises.

### Points d'attention pour le notebook de modelisation
- Suivre les **faux negatifs**, car ils representent les profils malades non detectes.
- Ne pas se limiter a l'accuracy : utiliser aussi le rappel, la specificite, la PR-AUC et les matrices de confusion.
- Pour `heart`, l'encodage de la cible est inverse : `target = 1` signifie patient sain. Les metriques medicales doivent donc considerer `target = 0` comme la classe malade.

---
*Suite : voir le notebook `02_modelisation_prediction.ipynb` pour la phase de modelisation.*
